# Module 06: Resource Teardown and Cost Management
In this final module, you will clean up all deployed Google Cloud resources to prevent unexpected compute billing.

### Cleanup Steps:
1. Delete active Kubernetes JobSets (`jax-cpu-job`, `jax-scale-job`, `jax-gpu-job`, `jax-tpu-job`).
2. Optional: Delete TPU and GPU accelerator node pools.
3. Scale default CPU node pool back down to 1 node.
4. Delete Google Artifact Registry images and repository.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import config
cfg = config.load_config("../config.env")

PROJECT_ID = cfg["PROJECT_ID"]
REGION = cfg["REGION"]
ZONE = cfg["ZONE"]
CLUSTER_NAME = cfg["CLUSTER_NAME"]
REPO = cfg["ARTIFACT_REGISTRY_REPO"]
TPU_NODE_POOL = cfg["TPU_NODE_POOL_NAME"]
GPU_NODE_POOL = cfg["GPU_NODE_POOL_NAME"]

## 1. Delete Kubernetes JobSets

In [ ]:
!kubectl delete jobset jax-cpu-job jax-scale-job jax-gpu-job jax-tpu-job --ignore-not-found

## 2. Scale Default CPU Pool Back Down to 1 Node

In [ ]:
!gcloud container clusters resize {CLUSTER_NAME} \
    --node-pool=default-pool \
    --num-nodes=1 \
    --zone={ZONE} \
    --quiet

## 3. Delete Accelerator Node Pools (If Created)

In [ ]:
!gcloud container node-pools delete {TPU_NODE_POOL} --cluster={CLUSTER_NAME} --zone={ZONE} --quiet

In [ ]:
!gcloud container node-pools delete {GPU_NODE_POOL} --cluster={CLUSTER_NAME} --zone={ZONE} --quiet

## 4. Delete Artifact Registry Repository

In [ ]:
!gcloud artifacts repositories delete {REPO} --location={REGION} --quiet

## 5. Delete GKE Cluster & Custom VPC Subnet (Complete Cleanup)

In [ ]:
!gcloud container clusters delete {CLUSTER_NAME} --zone={ZONE} --quiet

In [ ]:
!gcloud compute networks subnets delete {cfg['SUBNET_NAME']} --region={REGION} --quiet

In [ ]:
!gcloud compute networks delete {cfg['NETWORK_NAME']} --quiet

## 6. Final GCP Status Check

In [ ]:
print("Teardown Complete! Verifying active cluster status:")
!gcloud container clusters list --zone={ZONE}